<a href="https://colab.research.google.com/github/Leticiavalcan/Leticiavalcan/blob/main/principais_pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# instalar pyspark via pip mais rapido
!pip install pyspark

In [ ]:
#Importando o pacote necessário para iniciar uma seção spark

from pyspark.sql import SparkSession

#Iniciando o spark context( nesta seção no santander onde esta local vou ter q colocar quanto de ram vou utilizar)
sc= SparkSession.builder.master('local[*]').getOrCreate()

In [ ]:
#iMPORTAÇÕES OBRIGATÓRIAS

from pyspark.sql.functions import col
from pyspark.sql.functions import sum, max, min, avg
from pyspark.sql.functions import lit
from pyspark.sql.functions import *

In [ ]:
# baixar o arquivo CSV a ser processado, so vai ser usado quando o arquivo esta em outra maquina e não na minha, é fazer um dowload para minha maquina
# baixar arquivo COVID
!wget --verbose --show-progress --no-check-certificate https://cursos.grandeporte.com.br/files/brq/big-data/datasets/covid_19_data.csv

In [ ]:
# fazer a leitura do arquivo CSV da máquina do Google através do pySpark

dataCovid = sc.read.csv('/content/covid_19_data.csv.1', inferSchema=True, header=True)

In [ ]:
#Visualização do dataframe
dataCovid.show(3)

In [ ]:
#Da o valor de analise estatisticas como media, soma, min, max
dfnomeie.describe().show()

In [ ]:
 # verificar se existe colunas nulos na coluna de países (Country/Region)
# o / serve para quebrar linha mas dizer ao phyton que deve executar como se fosse uma unica linha

dfCovid.filter(col('Country/Region').isNull()) \
    .count()

In [ ]:
#filtrando pais 'us' e o estado 'washinton', vemos nos casos confirmados que os valores só crescem
dataCovid.filter( " `Country/Region` = 'US' and `Province/State` =  'Washington' "  ).show(20)

In [ ]:
#agrupando por pais, depois somando e ordenando os casos confirmados em ordem decrescente
dataCovid.groupBy('Country/Region') \
            .sum() \
            .orderBy( col('sum(Confirmed)').desc() ) \
            .show(15)


In [ ]:
#1- remover tds as linhas de valores nulos
dataCovid.dropna().show(10)

In [ ]:
#2- preencher os valores nulos com outro valor de interesse
#comando fillna é usado para preencher os valores ausentes
dfCovid = dfCovid.fillna( value= 'N/A', subset=['Province/State'])
dfCovid.filter( col('Province/State') == 'N/A' ) \
                          .show(5)

In [ ]:
# como salvar os dados em um outro CSV

dataCovidFillNA.write.json('/content/null-estrategia-csv')

# como salvar os dados em um outro JSON

dataCovidFillNA.write.json('/content/null-estrategia-json')

In [ ]:
#encontrando a província que possui a maior quantidade de casos confirmados de covid

dfCovid=dfCovid.groupBy('Province/State')\
            .sum() \
            .orderBy( col('sum(Confirmed)').desc() )\
            .show(5)

In [ ]:
# devemos criar uma nova coluna para transformar a data em texto (ObservationDate) para uma data que o pyspark reconheça (ObservationDateNew)
dataCovid = dataCovid.withColumn('ObservationDateNew', to_date(dataCovid['ObservationDate'], 'MM/dd/yyyy'))

dataCovid = dataCovid.withColumn('Year', year( dataCovid['ObservationDateNew'] ))
dataCovid = dataCovid.withColumn('Month', month( dataCovid['ObservationDateNew'] ))
dataCovid = dataCovid.withColumn('Day', dayofmonth( dataCovid['ObservationDateNew'] ))

dataCovid.show(5)

In [ ]:
#.ALIAS altera o nome da coluna sem  alterar o tipo e os dados
dataCovidRenamed = dataCovid.select(
    col('SNo').alias('s_no'),
    col('ObservationDate').alias('observation_date'),
    col('Province/State').alias('province_state'),
    col("Country/Region").alias("country_region"),
    col("Last Update").alias("last_update") ,
    col("Confirmed").alias("confirmed") ,
    col("Deaths").alias("deaths")  ,
    col("ObservationDate").alias("observation_date_new")  ,
    col("Recovered").alias("recovered"),
    col("Year").alias("year"),
    col("Month").alias("month"),
    col("Day").alias("day")
)
dataCovidRenamed.show(3)

In [ ]:
#como deletar uma coluna do dataframe

dataCovidRenamed = dataCovidRenamed.drop('s_no')

dataCovidRenamed.show(3)